# OpenSource Clipping: Google Colab

Select **Runtime > Change runtime type > GPU** before starting. GPU hardware varies by session. Run these cells in order. Paste your YouTube URL in the options cell.

`/content` is temporary. A Python process interruption can resume from files that remain in this runtime. A disconnected or reset runtime may lose them. Optional Drive mounting below is for copying finished results; this notebook does not guarantee recovery from a full runtime reset.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/projectvault000/opensource-clipping.git'
REPO = Path('/content/opensource-clipping')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
elif not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not the project repository. Choose another directory.')
os.chdir(REPO)
print(f'Using repository: {REPO}')


In [ ]:
import shutil
import sys

if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)
if not shutil.which('deno'):
    subprocess.run(['bash', '-lc', 'set -o pipefail; curl -fsSL https://deno.land/install.sh | sh'], check=True)
    os.environ['PATH'] = str(Path.home() / '.deno' / 'bin') + os.pathsep + os.environ['PATH']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.DEVNULL)
subprocess.run(['ffprobe', '-version'], check=True, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, 'main.py', '--help'], check=True, stdout=subprocess.DEVNULL)
print('Dependencies, FFmpeg, FFprobe and CLI are available.')


## Colab Secrets
Add `GOOGLE_API_KEY` using the key icon in the Colab sidebar and allow notebook access. Add `PEXELS_API_KEY` only when enabling B-roll. Secret values are kept in process environment variables and are never printed.


In [ ]:
from google.colab import userdata

def colab_secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

google_key = colab_secret('GOOGLE_API_KEY')
if not google_key:
    raise RuntimeError('GOOGLE_API_KEY is missing from Colab Secrets or notebook access is disabled.')
os.environ['GOOGLE_API_KEY'] = google_key
for name in ('PEXELS_API_KEY', 'HF_TOKEN'):
    value = colab_secret(name)
    if value:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)
print('Colab Secrets configured.')


## Optional Drive copy
Keep the active render and temporary files in `/content`. Set `USE_DRIVE` to `True` to mount Drive and copy the approved ZIP after export. This does not make an interrupted job recoverable after a full runtime reset.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    print('Drive mounted for finished package copy.')
else:
    print('Working in temporary /content storage.')


## Job options
Start with one clip. After inspecting it, set `TOTAL_CLIPS = 5` and `VOICEOVER = True` for the production configuration. Changing options creates a different job ID; rerunning with the same options reuses the same job directory. If GPU is unavailable, set `WHISPER_DEVICE = 'cpu'` and `WHISPER_COMPUTE_TYPE = 'float32'`.


In [ ]:
VIDEO_URL = 'https://www.youtube.com/watch?v=YOUR_VIDEO_ID'
TOTAL_CLIPS = 1
WHISPER_MODEL = 'large-v3'
WHISPER_DEVICE = 'cuda'
WHISPER_COMPUTE_TYPE = 'float16'
USE_BROLL = False
VOICEOVER = False
COOKIE_FILE = ''  # Optional path to an uploaded cookies.txt

if 'YOUR_VIDEO_ID' in VIDEO_URL or not VIDEO_URL.startswith(('https://', 'http://')):
    raise ValueError('Set VIDEO_URL to a real video URL before running the job.')
if TOTAL_CLIPS < 1:
    raise ValueError('TOTAL_CLIPS must be at least 1.')
if USE_BROLL and not os.environ.get('PEXELS_API_KEY'):
    raise RuntimeError('PEXELS_API_KEY is missing. Disable B-roll or add it to Colab Secrets.')
if WHISPER_DEVICE == 'cuda':
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('GPU is unavailable. Enable a Colab GPU or use CPU with float32 Whisper.')
elif WHISPER_COMPUTE_TYPE == 'float16':
    raise ValueError('float16 Whisper requires a GPU. Select float32 for CPU.')
disk = shutil.disk_usage('/content')
print(f'Free /content space: {disk.free / (1024 ** 3):.1f} GiB')
print(f'Job: {TOTAL_CLIPS} clip(s), Whisper {WHISPER_DEVICE}/{WHISPER_COMPUTE_TYPE}')


In [ ]:
from clipping.config import build_config
from clipping.checkpoints import job_id_for_config

args = ['--url', VIDEO_URL, '--clips', str(TOTAL_CLIPS), '--ratio', '9:16',
        '--font-style', 'HORMOZI', '--hook-duration', '3', '--words-per-sub', '5',
        '--whisper-model', WHISPER_MODEL, '--whisper-device', WHISPER_DEVICE,
        '--whisper-compute-type', WHISPER_COMPUTE_TYPE, '--no-bgm']
if not USE_BROLL:
    args.append('--no-broll')
if VOICEOVER:
    args += ['--voiceover', '--voiceover-style', 'reaction',
             '--voiceover-length', 'normal', '--voiceover-voice', 'en-US-AvaNeural']
if COOKIE_FILE:
    cookie_path = Path(COOKIE_FILE).expanduser().resolve()
    if not cookie_path.is_file():
        raise FileNotFoundError(f'Cookie file not found: {cookie_path}')
    args += ['--yt-cookies', str(cookie_path)]
CURRENT_JOB_ID = job_id_for_config(build_config(args))
CURRENT_JOB_DIR = REPO / 'outputs' / CURRENT_JOB_ID
print(f'Job ID: {CURRENT_JOB_ID}')
subprocess.run([sys.executable, 'main.py', *args], check=True)


In [ ]:
import json

manifest_path = CURRENT_JOB_DIR / 'render_manifest.json'
if not manifest_path.is_file():
    raise RuntimeError(f'No render manifest for {CURRENT_JOB_ID}; inspect the pipeline error above.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for row in manifest:
    video = Path(row.get('video_path') or '')
    print(row.get('clip_id'), row.get('quality_status'), row.get('review_status'),
          str(video) if video.is_file() else 'VIDEO MISSING')
print(f'{len(manifest)} manifest row(s) for {CURRENT_JOB_ID}')


## Review clips
Preview each clip and inspect its QC warnings before approval. Set decisions in the next cell (`APPROVED`, `REJECTED`, or `NEEDS_CHANGES`). A clip marked `NEEDS_CHANGES` can be rerendered below. To change its selection or text, edit that rank in the job's `metadata_preview.json` first. Keep the source video in `/content` for the fastest retry.


In [ ]:
from IPython.display import Video, display
from clipping.review_manager import ReviewManager

manager = ReviewManager(CURRENT_JOB_DIR)
print(manager.build_status_lines())
PREVIEW_CLIP_ID = 'clip_01'
row = manager.get_clip(PREVIEW_CLIP_ID)
preview_path = Path(row.get('video_path') or '')
if preview_path.is_file():
    display(Video(str(preview_path), embed=True, width=360))
else:
    print(f'No video to preview for {PREVIEW_CLIP_ID}.')


In [ ]:
REVIEW_DECISIONS = {}  # Example: {'clip_01': 'APPROVED'}
for clip_id, decision in REVIEW_DECISIONS.items():
    if decision not in {'APPROVED', 'REJECTED', 'NEEDS_CHANGES'}:
        raise ValueError(f'Unsupported decision for {clip_id}: {decision}')
    manager.set_review_status(clip_id, decision)
print(manager.build_status_lines())


In [ ]:
RETRY_CLIP_ID = ''  # Example: 'clip_01' after marking it NEEDS_CHANGES
if RETRY_CLIP_ID:
    subprocess.run([sys.executable, 'main.py', '--retry-clip', CURRENT_JOB_ID, RETRY_CLIP_ID], check=True)
    manager = ReviewManager(CURRENT_JOB_DIR)
    print(manager.build_status_lines())
    print('Inspect the new video and QC, then approve this generation explicitly.')


In [ ]:
from zipfile import ZipFile
from google.colab import files

result = manager.export_approved_set()
archive_path = Path(result['zip_path'])
with ZipFile(archive_path) as archive:
    names = archive.namelist()
if not result['approved'] or not names:
    raise RuntimeError('No approved clips were packaged. Review and approve at least one passing clip.')
print(f'Approved package: {archive_path} ({len(names)} files)')
if USE_DRIVE:
    destination = Path('/content/drive/MyDrive/clipping_results')
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive_path, destination / f'{CURRENT_JOB_ID}_approved_clips.zip')
files.download(str(archive_path))
